# 03 Ji Analysis

仮説 H1–H4 を検証するための分析ノートブックです。

- H1: 緑重視（`w2`↑）で緑地周辺の順位が上がる
- H2: 15 時の `J_i` が 8 時より大きい
- H3: POI 欠損区域では `C` の分散が小さい
- H4: 猛暑日プリセットで WBGT 寄与が増える

In [3]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path
from random import Random

import pandas as pd

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook fallback
    display = print

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / 'data').exists() and (candidate / 'src').exists():
        REPO_ROOT = candidate
        break

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from heat_town.model import compute_contributions, compute_ji, normalize_weights

DATA_DIRS = [REPO_ROOT / 'data' / 'samples', REPO_ROOT / 'data' / 'processed']
SUPPORTED_SUFFIXES = {'.parquet', '.csv', '.json', '.geojson'}

def read_geojson(path: Path) -> pd.DataFrame:
    payload = json.loads(path.read_text(encoding='utf-8'))
    features = payload.get('features', []) if isinstance(payload, dict) else []
    rows: list[dict[str, object]] = []
    for feature in features:
        properties = dict(feature.get('properties') or {})
        geometry = feature.get('geometry') or {}
        properties['geometry_type'] = geometry.get('type')
        properties['geometry'] = geometry.get('coordinates')
        rows.append(properties)
    return pd.DataFrame(rows)

def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.json', '.geojson'}:
        return read_geojson(path)
    raise ValueError(f'Unsupported file: {path}')

def discover_frames() -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for directory in DATA_DIRS:
        if not directory.exists():
            continue
        for path in sorted(directory.rglob('*')):
            if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES:
                try:
                    frames[path.stem] = read_table(path)
                except Exception:
                    continue
    return frames

def pick_column(frame: pd.DataFrame, candidates: list[str]) -> str | None:
    lowered = {str(column).lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    for column in frame.columns:
        column_lower = str(column).lower()
        if any(candidate.lower() in column_lower for candidate in candidates):
            return column
    return None

def parse_hour(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        return series.dt.hour
    numeric = pd.to_numeric(series, errors='coerce')
    if numeric.notna().any() and numeric.dropna().between(0, 23).all():
        return numeric.astype('Int64')
    parsed = pd.to_datetime(series, errors='coerce')
    return parsed.dt.hour

def bootstrap_mean_difference(left: pd.Series, right: pd.Series, iterations: int = 2000, seed: int = 42) -> tuple[float, tuple[float, float]]:
    left_values = pd.to_numeric(left, errors='coerce').dropna().tolist()
    right_values = pd.to_numeric(right, errors='coerce').dropna().tolist()
    if len(left_values) == 0 or len(right_values) == 0:
        return float('nan'), (float('nan'), float('nan'))
    observed = float(sum(left_values) / len(left_values) - sum(right_values) / len(right_values))
    rng = Random(seed)
    diffs = []
    for _ in range(iterations):
        sample_left = [rng.choice(left_values) for _ in range(len(left_values))]
        sample_right = [rng.choice(right_values) for _ in range(len(right_values))]
        diffs.append(sum(sample_left) / len(sample_left) - sum(sample_right) / len(sample_right))
    diffs.sort()
    lower_index = max(0, int(0.025 * (len(diffs) - 1)))
    upper_index = min(len(diffs) - 1, int(0.975 * (len(diffs) - 1)))
    return observed, (float(diffs[lower_index]), float(diffs[upper_index]))

frames = discover_frames()
print(f'loaded {len(frames)} frame(s)')
for name, frame in frames.items():
    print(f'- {name}: {frame.shape}')

loaded 0 frame(s)


In [7]:
def select_feature_frame(frames: dict[str, pd.DataFrame]) -> tuple[str | None, pd.DataFrame | None]:
    if not frames:
        return None, None
    ranked: list[tuple[tuple[int, int, int], str, pd.DataFrame]] = []
    for name, frame in frames.items():
        columns_lower = {str(column).lower() for column in frame.columns}
        has_core = int({'d', 'comfort', 'wbgt'}.issubset(columns_lower))
        numeric_count = len(frame.select_dtypes(include='number').columns)
        ranked.append(((has_core, numeric_count, len(frame)), name, frame))
    ranked.sort(reverse=True)
    _, name, frame = ranked[0]
    return name, frame

frame_name, frame = select_feature_frame(frames)
if frame is None:
    print('No readable data found yet.')
else:
    print(f'analysis target: {frame_name}')
    display(frame.head(10))

    distance_col = pick_column(frame, ['d', 'distance'])
    comfort_col = pick_column(frame, ['c', 'comfort'])
    wbgt_col = pick_column(frame, ['wbgt'])
    ji_col = pick_column(frame, ['ji', 'j_i', 'score'])
    green_col = pick_column(frame, ['green', 'park', 'shade', 'tree', 'vegetation'])
    poi_missing_col = pick_column(frame, ['poi_missing', 'missing_poi', 'has_poi'])
    time_col = pick_column(frame, ['time', 'timestamp', 'datetime', 'hour'])
    alert_col = pick_column(frame, ['heat_alert', 'heatwave', 'hot_day', 'preset'])

    print('distance:', distance_col)
    print('comfort:', comfort_col)
    print('wbgt:', wbgt_col)
    print('ji:', ji_col)
    print('green proxy:', green_col)
    print('poi missing:', poi_missing_col)
    print('time:', time_col)
    print('alert:', alert_col)

No readable data found yet.


In [8]:
results: list[dict[str, object]] = []

if frame is not None and all(column is not None for column in [distance_col, comfort_col, wbgt_col]):
    analysis = frame[[column for column in [distance_col, comfort_col, wbgt_col, ji_col, green_col, poi_missing_col, time_col, alert_col] if column is not None]].copy()
    analysis = analysis.rename(columns={distance_col: 'distance', comfort_col: 'comfort', wbgt_col: 'wbgt'})
    if ji_col is None:
        base_w = normalize_weights(0.3, 0.4, 0.3)
        analysis['ji'] = analysis.apply(lambda row: compute_ji(row['distance'], row['comfort'], row['wbgt'], *base_w), axis=1)
    else:
        analysis = analysis.rename(columns={ji_col: 'ji'})

    if green_col is not None:
        analysis = analysis.rename(columns={green_col: 'green_proxy'})
    if poi_missing_col is not None:
        analysis = analysis.rename(columns={poi_missing_col: 'poi_missing'})
    if time_col is not None:
        analysis = analysis.rename(columns={time_col: 'time_value'})
    if alert_col is not None:
        analysis = analysis.rename(columns={alert_col: 'alert_value'})

    analysis = analysis.dropna(subset=['distance', 'comfort', 'wbgt', 'ji']).copy()

    base_w = normalize_weights(0.3, 0.4, 0.3)
    green_w = normalize_weights(0.2, 0.6, 0.2)
    hot_w = normalize_weights(0.2, 0.3, 0.5)

    if 'green_proxy' in analysis.columns and analysis['green_proxy'].notna().any():
        baseline = analysis[['distance', 'comfort', 'wbgt']].apply(lambda row: compute_ji(row['distance'], row['comfort'], row['wbgt'], *base_w), axis=1)
        green_heavy = analysis[['distance', 'comfort', 'wbgt']].apply(lambda row: compute_ji(row['distance'], row['comfort'], row['wbgt'], *green_w), axis=1)
        rank_gain = baseline.rank(method='average') - green_heavy.rank(method='average')
        green_top = analysis['green_proxy'].rank(pct=True) >= 0.75
        observed, interval = bootstrap_mean_difference(rank_gain[green_top], rank_gain[~green_top])
        results.append({'hypothesis': 'H1', 'status': 'checked', 'metric': 'rank gain, green-heavy minus baseline', 'value': observed, 'interval': interval, 'interpretation': 'positive means greener rows improve relative rank when w2 increases'})
    else:
        results.append({'hypothesis': 'H1', 'status': 'skipped', 'metric': 'green proxy column not found', 'value': None, 'interval': None, 'interpretation': 'needs a green/park/shade/tree feature'})

    if 'time_value' in analysis.columns:
        hour = parse_hour(analysis['time_value'])
        analysis = analysis.assign(hour=hour)
        ji_15 = analysis.loc[analysis['hour'] == 15, 'ji']
        ji_8 = analysis.loc[analysis['hour'] == 8, 'ji']
        if len(ji_15) and len(ji_8):
            observed, interval = bootstrap_mean_difference(ji_15, ji_8)
            results.append({'hypothesis': 'H2', 'status': 'checked', 'metric': 'mean J_i difference, 15h minus 8h', 'value': observed, 'interval': interval, 'interpretation': 'positive supports 15h J_i > 8h J_i'})
        else:
            results.append({'hypothesis': 'H2', 'status': 'skipped', 'metric': 'insufficient 8h/15h coverage', 'value': None, 'interval': None, 'interpretation': 'needs enough rows at both 8h and 15h'})
    else:
        results.append({'hypothesis': 'H2', 'status': 'skipped', 'metric': 'time column not found', 'value': None, 'interval': None, 'interpretation': 'needs hour or timestamp information'})

    if 'poi_missing' in analysis.columns and analysis['poi_missing'].notna().any():
        missing_mask = analysis['poi_missing'].astype(bool)
        missing_var = float(analysis.loc[missing_mask, 'comfort'].var(ddof=1))
        present_var = float(analysis.loc[~missing_mask, 'comfort'].var(ddof=1))
        ratio = missing_var / present_var if present_var != 0.0 and math.isfinite(present_var) else float('nan')
        results.append({'hypothesis': 'H3', 'status': 'checked', 'metric': 'variance ratio, POI-missing / POI-present', 'value': ratio, 'interval': (missing_var, present_var), 'interpretation': 'ratio below 1 supports smaller comfort variance in POI-missing areas'})
    else:
        results.append({'hypothesis': 'H3', 'status': 'skipped', 'metric': 'POI missing flag not found', 'value': None, 'interval': None, 'interpretation': 'needs a POI missing/presence indicator'})

    baseline_share = []
    hot_share = []
    for _, row in analysis[['distance', 'comfort', 'wbgt']].iterrows():
        base = compute_contributions(float(row['distance']), float(row['comfort']), float(row['wbgt']), *base_w)
        hot = compute_contributions(float(row['distance']), float(row['comfort']), float(row['wbgt']), *hot_w)
        baseline_share.append(base['heat'] / base['total'] if base['total'] else float('nan'))
        hot_share.append(hot['heat'] / hot['total'] if hot['total'] else float('nan'))
    baseline_share = pd.Series(baseline_share).dropna()
    hot_share = pd.Series(hot_share).dropna()
    if len(baseline_share) and len(hot_share):
        observed, interval = bootstrap_mean_difference(hot_share, baseline_share)
        results.append({'hypothesis': 'H4', 'status': 'checked', 'metric': 'WBGT share difference, hot-day minus baseline', 'value': observed, 'interval': interval, 'interpretation': 'positive means hot-day preset gives WBGT a larger contribution share'})
    else:
        results.append({'hypothesis': 'H4', 'status': 'skipped', 'metric': 'insufficient rows for contribution comparison', 'value': None, 'interval': None, 'interpretation': 'needs rows with distance, comfort, and wbgt'})
else:
    results.append({'hypothesis': 'H1-H4', 'status': 'skipped', 'metric': 'core feature columns not found', 'value': None, 'interval': None, 'interpretation': 'need distance/comfort/wbgt columns in the target frame'})

results_df = pd.DataFrame(results)
display(results_df)

,hypothesis,status,metric,value,interval,interpretation
0,H1-H4,skipped,core feature columns not found,None,None,need distance/comfort/wbgt columns in the targ...


In [9]:
if not results_df.empty:
    checked = results_df[results_df['status'] == 'checked'].copy()
    if not checked.empty:
        display(checked[['hypothesis', 'metric', 'value', 'interval', 'interpretation']])

    h2_rows = results_df[results_df['hypothesis'] == 'H2']
    if not h2_rows.empty and h2_rows.iloc[0]['status'] == 'checked':
        row = h2_rows.iloc[0]
        print(f"H2 result: {row['value']:.4f} (95% CI {row['interval'][0]:.4f} to {row['interval'][1]:.4f})")

    if (results_df['status'] == 'checked').any():
        print('At least one hypothesis has been evaluated.')
    else:
        print('No hypothesis could be fully evaluated with the currently available columns.')

No hypothesis could be fully evaluated with the currently available columns.
